# Topological Data Analysis: Persistent Homology on Molecular Structures

**Author:** Nandini  
**Stack:** GUDHI, RDKit, NumPy, matplotlib  
**Topics:** Vietoris-Rips filtration on 3D conformers, persistence diagrams, Betti curves, bottleneck distance between molecules

---

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import AllChem
import gudhi
from gudhi.representations import BettiCurve, PersistenceImage
print(f'GUDHI version: {gudhi.__version__}')
print('All imports OK')

GUDHI version: 3.12.0
All imports OK


## 1. Generate 3D Conformers from SMILES
We embed molecules in 3D using ETKDG and extract atomic coordinates as point clouds.

In [2]:
molecules = {
    'Aspirin': 'CC(=O)OC1=CC=CC=C1C(=O)O',
    'Ibuprofen': 'CC(C)CC1=CC=C(C=C1)C(C)C(=O)O',
    'Caffeine': 'CN1C=NC2=C1C(=O)N(C(=O)N2C)C',
    'Celecoxib': 'CC1=CC=C(C=C1)C1=CC(=NN1C1=CC=C(C=C1)S(N)(=O)=O)C(F)(F)F',
    'Naproxen': 'COC1=CC2=CC(=CC2=CC1)C(C)C(=O)O',
}

def smiles_to_point_cloud(smi, seed=42):
    mol = Chem.MolFromSmiles(smi)
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = seed
    AllChem.EmbedMolecule(mol, params)
    AllChem.MMFFOptimizeMolecule(mol)
    conf = mol.GetConformer()
    coords = np.array([list(conf.GetAtomPosition(i)) for i in range(mol.GetNumAtoms())])
    # Heavy atoms only
    heavy_idx = [i for i in range(mol.GetNumAtoms()) if mol.GetAtomWithIdx(i).GetAtomicNum() > 1]
    return coords[heavy_idx], mol

point_clouds = {}
for name, smi in molecules.items():
    pc, mol = smiles_to_point_cloud(smi)
    point_clouds[name] = pc
    print(f'{name:12s}: {pc.shape[0]} heavy atoms, shape {pc.shape}')

Aspirin     : 13 heavy atoms, shape (13, 3)
Ibuprofen   : 15 heavy atoms, shape (15, 3)
Caffeine    : 14 heavy atoms, shape (14, 3)


Celecoxib   : 26 heavy atoms, shape (26, 3)
Naproxen    : 16 heavy atoms, shape (16, 3)


## 2. Vietoris-Rips Persistence Diagrams
Computing persistent homology (H0 = connected components, H1 = loops, H2 = voids) on molecular point clouds.

In [3]:
def compute_persistence(point_cloud, max_edge=8.0, max_dim=2):
    """Compute Vietoris-Rips persistence on a 3D point cloud."""
    rips = gudhi.RipsComplex(points=point_cloud, max_edge_length=max_edge)
    st = rips.create_simplex_tree(max_dimension=max_dim + 1)
    st.compute_persistence()
    return st

persistence_data = {}
for name, pc in point_clouds.items():
    st = compute_persistence(pc)
    diag = st.persistence()
    persistence_data[name] = st
    
    # Count features per dimension
    for dim in range(3):
        pairs = st.persistence_intervals_in_dimension(dim)
        finite = pairs[pairs[:, 1] != np.inf] if len(pairs) > 0 else np.array([])
        n_total = len(pairs)
        n_finite = len(finite)
        print(f'{name:12s} H{dim}: {n_total} features ({n_finite} finite)')

Aspirin      H0: 13 features (12 finite)
Aspirin      H1: 1 features (1 finite)
Aspirin      H2: 1 features (1 finite)
Ibuprofen    H0: 15 features (14 finite)
Ibuprofen    H1: 1 features (1 finite)
Ibuprofen    H2: 1 features (1 finite)
Caffeine     H0: 14 features (13 finite)
Caffeine     H1: 2 features (2 finite)
Caffeine     H2: 1 features (1 finite)
Celecoxib    H0: 26 features (25 finite)
Celecoxib    H1: 3 features (3 finite)
Celecoxib    H2: 2 features (2 finite)
Naproxen     H0: 16 features (15 finite)
Naproxen     H1: 2 features (2 finite)
Naproxen     H2: 1 features (1 finite)


In [4]:
# Plot persistence diagrams for all molecules
fig, axes = plt.subplots(1, len(molecules), figsize=(4*len(molecules), 4))
colors = {0: '#ef4444', 1: '#3b82f6', 2: '#22c55e'}

for idx, (name, st) in enumerate(persistence_data.items()):
    ax = axes[idx]
    max_val = 0
    for dim in range(3):
        pairs = st.persistence_intervals_in_dimension(dim)
        if len(pairs) > 0:
            finite = pairs[pairs[:, 1] != np.inf]
            if len(finite) > 0:
                ax.scatter(finite[:, 0], finite[:, 1], s=15, alpha=0.7,
                          color=colors[dim], label=f'H{dim}', zorder=3)
                max_val = max(max_val, finite.max())
    
    lim = max_val * 1.1 if max_val > 0 else 8
    ax.plot([0, lim], [0, lim], 'k--', alpha=0.3, linewidth=0.8)
    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_xlabel('Birth', fontsize=8)
    if idx == 0:
        ax.set_ylabel('Death', fontsize=8)
    ax.legend(fontsize=7, loc='lower right')
    ax.set_aspect('equal')

plt.suptitle('Persistence Diagrams from Vietoris-Rips on Molecular 3D Conformers', y=1.02)
plt.tight_layout()
plt.show()

## 3. Betti Curves as Molecular Descriptors
Vectorizing persistence diagrams into fixed-length Betti curve representations for ML downstream.

In [5]:
bc = BettiCurve(resolution=100, sample_range=[0, 8])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
filtration_values = np.linspace(0, 8, 100)

for dim, ax in enumerate(axes):
    for name, st in persistence_data.items():
        pairs = st.persistence_intervals_in_dimension(dim)
        if len(pairs) > 0:
            # Manual Betti curve computation
            betti = np.zeros(100)
            for b, d in pairs:
                if d == np.inf:
                    d = 8.0
                mask = (filtration_values >= b) & (filtration_values < d)
                betti[mask] += 1
            ax.plot(filtration_values, betti, label=name, linewidth=1.5)
    
    ax.set_xlabel('Filtration value (Angstrom)')
    ax.set_ylabel(f'Betti-{dim}')
    ax.set_title(f'H{dim} Betti Curves')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

plt.suptitle('Betti Curves as Topological Fingerprints of Drug Molecules')
plt.tight_layout()
plt.show()

## 4. Bottleneck Distance Between Molecules
Using the bottleneck distance on H1 persistence diagrams as a topological similarity metric.

In [6]:
names_list = list(persistence_data.keys())
n = len(names_list)
dist_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(i+1, n):
        diag_i = persistence_data[names_list[i]].persistence_intervals_in_dimension(1)
        diag_j = persistence_data[names_list[j]].persistence_intervals_in_dimension(1)
        # Filter to finite intervals
        di = diag_i[diag_i[:, 1] != np.inf] if len(diag_i) > 0 else np.array([]).reshape(0, 2)
        dj = diag_j[diag_j[:, 1] != np.inf] if len(diag_j) > 0 else np.array([]).reshape(0, 2)
        
        if len(di) > 0 and len(dj) > 0:
            d = gudhi.bottleneck_distance(di.tolist(), dj.tolist())
        else:
            d = 0.0
        dist_matrix[i, j] = d
        dist_matrix[j, i] = d

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(dist_matrix, cmap='magma_r')
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(names_list, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(names_list, fontsize=9)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{dist_matrix[i,j]:.2f}', ha='center', va='center', fontsize=7,
                color='white' if dist_matrix[i,j] > dist_matrix.max()/2 else 'black')
plt.colorbar(im, label='Bottleneck Distance (H1)')
ax.set_title('Topological Distance Between Drug Molecules (H1)')
plt.tight_layout()
plt.show()

print('\nInterpretation: Bottleneck distance on H1 captures differences in ring/loop topology.')
print('Molecules with similar ring systems have small bottleneck distance.')


Interpretation: Bottleneck distance on H1 captures differences in ring/loop topology.
Molecules with similar ring systems have small bottleneck distance.
